# Hellinger distance

The Hellinger distnace for two discrete distributions $p = (p_0, \dots , p_n)$ and $q = (q_0, \dots, q_n) $ is defined as follows:

$$ H(p, q) = \frac{1}{\sqrt{2}} \sqrt{\sum_i^n (\sqrt{p_i} - \sqrt{q_i})^2}$$

In [5]:
import numpy as np
from scipy.stats import dirichlet
import matplotlib.pyplot as plt
import math
import time
import torch
from torchvision.datasets import CIFAR10, MNIST
import torch.nn.functional as F

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
#CIFAR LOAD
cifar = CIFAR10(root='data', train=True, download=True)
data_np_cifar = cifar.data.astype(np.float32) / 255.0
data_np_cifar = np.transpose(data_np_cifar, (0, 3, 1, 2))  
#MNIST LOAD
mnist = MNIST(root='data', train=True, download=True)
data_np_mnist = mnist.data.numpy().astype(np.float32) / 255.0
data_np_mnist = data_np_mnist[:, None, :, :]  
mnist_tensor = torch.from_numpy(data_np_mnist)
mnist_tensor = F.interpolate(mnist_tensor, size=(32, 32), mode='bilinear', align_corners=False)
mnist_tensor = mnist_tensor.repeat(1, 3, 1, 1)
data_np_mnist = mnist_tensor[:50000].numpy()
print("Cfiar shape:", data_np_cifar.shape)
print("Mnist shape:", data_np_mnist.shape)

Cfiar shape: (50000, 3, 32, 32)
Mnist shape: (50000, 3, 32, 32)


In [7]:
flattened_cifar = data_np_cifar.reshape(50000, 1, 3072, 1)
flattened_mnist = data_np_mnist.reshape(50000, 1, 3072, 1)

## 1. Original version

In [ ]:
import numpy as np

def hellinger_np(p, q):
    """Efficient Hellinger distance between two discrete distributions using NumPy."""
    return np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))

def hellinger(p, q):
    """Hellinger distance between two discrete distributions.
       In pure Python.
       Original, ugly version.
    """
    return sum([(math.sqrt(t[0])-math.sqrt(t[1]))*(math.sqrt(t[0])-math.sqrt(t[1]))\
                for t in zip(p,q)])/math.sqrt(2.)

In [9]:
hellinger_np(flattened_cifar, flattened_cifar)


np.float32(0.0)

In [10]:
hellinger_np(flattened_cifar, flattened_mnist)


np.float32(5468.864)

Despite from being ugly, the version `hellinger(p, q)` ommits the square root of the sum. This is a legal thing to, but notice that in this case the maximum distance of $p$ and $q$ is $\sqrt{2}$ and not 1.

## 2. Some improvements

In [15]:
def hellinger2(p, q):
    """Hellinger distance between two discrete distributions. 
       In pure Python.
       Some improvements.
    """
    return math.sqrt(sum([ (math.sqrt(p_i) - math.sqrt(q_i))**2 for p_i, q_i in zip(p, q) ]) / 2)

def hellinger2_np(p, q):
    """Hellinger distance between two discrete distributions using NumPy."""
    return np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q))**2))


In `hellinger2(p, q)` I replace the the variable `t` with `p_i` and `q_i` for better readability. I further utilized the `**` in order to calculate the square, since it is much faster than calculating the square root for each `p_i` and `q_i` two times. The line continuation operator `\` is needless here, so it can be omitted. Since $\sqrt{\frac{x}{y}} = \frac{\sqrt{x}}{\sqrt{y}}$, we save one square root operation by first deviding the sum by 2 and than taking the square root of the result.

In [16]:
hellinger2_np(flattened_cifar, flattened_mnist)

np.float32(5468.864)

## 3. Without list comprehension

In [ ]:
def hellinger_explicit(p, q):
    """Hellinger distance between two discrete distributions.
       In pure Python.
       Same as hellinger2 but without list comprehension
    """
    list_of_squares = []
    for p_i, q_i in zip(p, q):

        # caluclate the square of the difference of ith distr elements
        s = (math.sqrt(p_i) - math.sqrt(q_i)) ** 2
        
        # append 
        list_of_squares.append(s)
    
    # calculate sum of squares
    sosq = sum(list_of_squares)    
    
    return math.sqrt(sosq / 2)

## 4. Further improvements

In [ ]:
def hellinger_dot(p, q):
    """Hellinger distance between two discrete distributions. 
       Using numpy.
       For Python >= 3.5 only"""
    z = np.sqrt(p) - np.sqrt(q)
    return np.sqrt(z @ z / 2)

The sum of squares of the elements of some vector is nothing else but a dot product. Thus, asuming `p` and `q` to be numpy arrays, we can utilize matrix product operator `@` from Python >= 3.5, which very much simplifies the code.

## 5. Need for speed

In [ ]:
def hellinger_fast(p, q):
    """Hellinger distance between two discrete distributions.
       In pure Python.
       Fastest version.
    """
    return sum([ (math.sqrt(p_i) - math.sqrt(q_i))**2 for p_i, q_i in zip(p,q) ])

We can slightly improve the performance of the pure python versions. Notice that, as in `hellinger(p, q)` on can ommit the sqaure root since it is a monoonic function. On could also ommit the factor $\frac{1}{\sqrt{2}}$ because it only scales the sum. This, however, that for two maximally different distribution, maximum value of `hellinger_fast(p, q)` is two.  

## 6. Time it!

In [ ]:
%timeit hellinger(x, y)

In [ ]:
%timeit hellinger2(x, y)

In [ ]:
%timeit hellinger_explicit(x, y)

In [ ]:
%timeit hellinger_dot(x, y)

In [ ]:
%timeit hellinger_fast(x, y)

## 7. Performance test with graphical output

In [ ]:
funs = {'hellinger':hellinger, 
        'hellinger2':hellinger2,
        'hellinger_explicit':hellinger_explicit, 
        'hellinger_fast':hellinger_fast,
        'hellinger_dot':hellinger_dot}

In [ ]:
MAX_ELEM = 1000
MAX_RUNS = 100

benchmark = np.empty((MAX_ELEM, len(funs), MAX_RUNS))

for i in range(MAX_ELEM):
    p, q = dirichlet.rvs([1]*(i+1), size=2)
    
    for j, fun in enumerate(funs.values()):
        
        for k in range(MAX_RUNS):
        
            start = time.time()
            _ = fun(p, q)
            benchmark[i, j, k] = time.time() - start

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(8, 5))
ax.plot(benchmark.mean(axis=2))            
ax.set_xlabel('Length of input vector')
ax.set_ylabel('Mean time in [s] over {} runs'.format(MAX_RUNS))
ax.legend(list(funs.keys()))
plt.show()

The original function scales worst. There is no big difference between `hellinger2` and `hellinger_fast` since they only differ by of division. `hellinger_explicit` scales worse since explicit for loops are always slower then list comprehensions. `hellinger_dot` outpreforms the previous functions, since the for loop could be ommitted.

Thus, for discrete distribution less then twelve entries the best choice is `hellinger2`. For distributions with more entries one should use the `hellinger_dot`. 